# Voter Turnout Prediction Model
## Overview
#### The goal of this project is to predict whether a voter who participated in the 2024 presidential election also participated in the Tempe municipal election.

#### Municipal elections typically have significantly lower turnout than presidential elections. Understanding which voters participate in both can help improve targeted outreach strategies and turnout prediction models.

#### The workflow for this project consists of:

#### 1. Data collection and merging

#### 2. Data cleaning and preprocessing

#### 3. Feature engineering

#### 4. Encoding categorical variables

#### 5. Model training (Logistic Regression)

#### 6. Model evaluation (confusion matrix and classification metrics)

#### 7. Interpretation of coefficients and odds ratios

#### 8. Zip-Code analysis

In [15]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np

In [16]:
df_pres = pd.read_excel("FINAL-VOTEDFILE_NOV 2024-TEMPE.xlsx")


df_city = pd.read_excel("FINAL-VOTEDFILE_MAR2024-TEMPE 03182024.xlsx")

In [17]:
df_city.head()

,VOTERID,ELECTNO,ELECTDATE,FIRSTNAME,MIDNAME,LASTNAME,HSENO,HSENOSUFX,STDIR,STNAME,...,CPC,PARTY,BALTYPE,CTYDIST,SCHDIST,PRIMBALTYPE,FEDERAL,AEVL,YOB,FEDBLTISSUED
0,10134,1402,20240312,DEBORAH,LEE,BALZER,1131.0,NaN,E,LEEWARD,...,5198,DEM,R,TE,3.0,NaN,N,Y,1954,N
1,10970,1402,20240312,JOHN,NATHAN,FIELDEN,916.0,NaN,E,VERDE,...,5199,DEM,R,TE,28.0,NaN,N,Y,1965,N
2,11212,1402,20240312,BRIAN,JOSEPH,GRATTON,425.0,NaN,W,RIO SALADO,...,5192,PND,R,TE,3.0,NaN,N,Y,1946,N
3,11790,1402,20240312,JANET,RENE,LANGE,425.0,NaN,W,RIO SALADO,...,5192,DEM,R,TE,3.0,NaN,N,Y,1946,N
4,11791,1402,20240312,MICHAEL,DAVID,LANGE,425.0,NaN,W,RIO SALADO,...,5192,DEM,R,TE,3.0,NaN,N,Y,1945,N


In [18]:
df_pres.head()

,VOTERID,ELECTNO,ELECTDATE,FIRSTNAME,MIDNAME,LASTNAME,HSENO,HSENOSUFX,STDIR,STNAME,...,PARTY,BALTYPE,CTYDIST,SCHDIST,PRIMBALTYPE,FEDERAL,AEVL,YOB,FEDBLTISSUED,Residency
0,1000669,1407,20241105,CAROL,M,TROGLIA,525,NaN,E,CITATION,...,REP,R,TE,28,NaN,N,Y,1962,N,
1,1000981,1407,20241105,COLLEEN,MARIE,HARRINGTON,2507,NaN,E,5TH,...,PND,P,TE,3,NaN,N,Y,1958,N,
2,1001379,1407,20241105,BETTY,LOUISE,EVANS,9646,NaN,S,DARROW,...,DEM,R,TE,28,NaN,N,Y,1934,N,
3,100186,1407,20241105,DAVID,NaN,ROSH,1859,NaN,E,OASIS,...,DEM,R,TE,28,NaN,N,N,1940,N,
4,1002217,1407,20241105,JANICE,LYNN,GIZA,8867,NaN,S,DATELAND,...,DEM,R,TE,28,NaN,N,Y,1957,N,


In [19]:
df_pres.info()
df_city.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74355 entries, 0 to 74354
Data columns (total 27 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   VOTERID       74355 non-null  int64  
 1   ELECTNO       74355 non-null  int64  
 2   ELECTDATE     74355 non-null  int64  
 3   FIRSTNAME     74350 non-null  object 
 4   MIDNAME       67048 non-null  object 
 5   LASTNAME      74354 non-null  object 
 6   HSENO         74355 non-null  int64  
 7   HSENOSUFX     24 non-null     object 
 8   STDIR         74355 non-null  object 
 9   STNAME        74355 non-null  object 
 10  STTYPE        73888 non-null  object 
 11  STSUFX        47 non-null     object 
 12  UNITNO        22706 non-null  object 
 13  CITY          74355 non-null  object 
 14  ZIPCODE       74355 non-null  int64  
 15  PHONENO       39100 non-null  float64
 16  CPC           74355 non-null  int64  
 17  PARTY         74355 non-null  object 
 18  BALTYPE       74355 non-nu

# Data cleaning
#### The raw voter files contained a large number of administrative fields, personally identifiable information, and variables that would introduce noise or data leakage. The dataset was cleaned in several steps to remove irrelevant variables and convert fields into a format usable for machine learning.


## Removing Personal Identificaiton Information
#### Several columns containing identifying information were removed because they provide no predictive value for turnout and introduce privacy concerns.


#### Dropped Columns:
#### Name - Phone Number - Adress Information - City(All voters are from Tempe, AZ)


## Data Leakage
#### Ballot Type, and whether the Federal Ballot was issued were removed as they are information that would not be available before the election, and irrelevant.


## Removing Redundant Election Variables
#### Election Date, and Election Number are redundant as the information is from the same election.


## Removing Structural Data 
#### City Precint District and School District were recently changed, and would not be able to be used for more recent election data



In [20]:
city_voters = df_city[['VOTERID']].drop_duplicates()
presidential_voters = df_pres.copy()

presidential_voters['voted_city'] = presidential_voters['VOTERID'].isin(city_voters['VOTERID']).astype(int)

In [21]:
presidential_voters.head()
presidential_voters.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74355 entries, 0 to 74354
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   VOTERID       74355 non-null  int64  
 1   ELECTNO       74355 non-null  int64  
 2   ELECTDATE     74355 non-null  int64  
 3   FIRSTNAME     74350 non-null  object 
 4   MIDNAME       67048 non-null  object 
 5   LASTNAME      74354 non-null  object 
 6   HSENO         74355 non-null  int64  
 7   HSENOSUFX     24 non-null     object 
 8   STDIR         74355 non-null  object 
 9   STNAME        74355 non-null  object 
 10  STTYPE        73888 non-null  object 
 11  STSUFX        47 non-null     object 
 12  UNITNO        22706 non-null  object 
 13  CITY          74355 non-null  object 
 14  ZIPCODE       74355 non-null  int64  
 15  PHONENO       39100 non-null  float64
 16  CPC           74355 non-null  int64  
 17  PARTY         74355 non-null  object 
 18  BALTYPE       74355 non-nu

In [22]:
#All of the voters are from the same city - Tempe
presidential_voters["CITY"].nunique()

1

In [23]:
voters1 = presidential_voters.drop(columns=["FIRSTNAME", "MIDNAME", "LASTNAME", "HSENO", "HSENOSUFX", "STDIR", "STNAME", "STTYPE", "STSUFX", "CITY","PHONENO", "UNITNO" ])

In [24]:
#Data Leakage
voters1 = voters1.drop(columns = ["BALTYPE", "FEDBLTISSUED"])

In [25]:
voters1.head()

,VOTERID,ELECTNO,ELECTDATE,ZIPCODE,CPC,PARTY,CTYDIST,SCHDIST,PRIMBALTYPE,FEDERAL,AEVL,YOB,Residency,voted_city
0,1000669,1407,20241105,85284,105,REP,TE,28,NaN,N,Y,1962,,1
1,1000981,1407,20241105,85288,872,PND,TE,3,NaN,N,Y,1958,,0
2,1001379,1407,20241105,85284,98,DEM,TE,28,NaN,N,Y,1934,,1
3,100186,1407,20241105,85283,162,DEM,TE,28,NaN,N,N,1940,,0
4,1002217,1407,20241105,85284,12,DEM,TE,28,NaN,N,Y,1957,,1


In [26]:
voters1["ELECTNO"].nunique()

1

In [27]:
voters1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74355 entries, 0 to 74354
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   VOTERID      74355 non-null  int64  
 1   ELECTNO      74355 non-null  int64  
 2   ELECTDATE    74355 non-null  int64  
 3   ZIPCODE      74355 non-null  int64  
 4   CPC          74355 non-null  int64  
 5   PARTY        74355 non-null  object 
 6   CTYDIST      74355 non-null  object 
 7   SCHDIST      74355 non-null  int64  
 8   PRIMBALTYPE  0 non-null      float64
 9   FEDERAL      74355 non-null  object 
 10  AEVL         74355 non-null  object 
 11  YOB          74355 non-null  int64  
 12  Residency    74355 non-null  object 
 13  voted_city   74355 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 7.9+ MB


In [28]:
id_col = voters1["VOTERID"]

In [29]:
#We merged so these variables are all the same - just create noise
voters2 = voters1.drop(columns = ["ELECTNO", "ELECTDATE"])

In [30]:
voters2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74355 entries, 0 to 74354
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   VOTERID      74355 non-null  int64  
 1   ZIPCODE      74355 non-null  int64  
 2   CPC          74355 non-null  int64  
 3   PARTY        74355 non-null  object 
 4   CTYDIST      74355 non-null  object 
 5   SCHDIST      74355 non-null  int64  
 6   PRIMBALTYPE  0 non-null      float64
 7   FEDERAL      74355 non-null  object 
 8   AEVL         74355 non-null  object 
 9   YOB          74355 non-null  int64  
 10  Residency    74355 non-null  object 
 11  voted_city   74355 non-null  int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 6.8+ MB


In [31]:
voters2.head()

,VOTERID,ZIPCODE,CPC,PARTY,CTYDIST,SCHDIST,PRIMBALTYPE,FEDERAL,AEVL,YOB,Residency,voted_city
0,1000669,85284,105,REP,TE,28,NaN,N,Y,1962,,1
1,1000981,85288,872,PND,TE,3,NaN,N,Y,1958,,0
2,1001379,85284,98,DEM,TE,28,NaN,N,Y,1934,,1
3,100186,85283,162,DEM,TE,28,NaN,N,N,1940,,0
4,1002217,85284,12,DEM,TE,28,NaN,N,Y,1957,,1


In [32]:
voters2 = voters2.drop(columns = ["Residency"])

In [33]:
voters2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74355 entries, 0 to 74354
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   VOTERID      74355 non-null  int64  
 1   ZIPCODE      74355 non-null  int64  
 2   CPC          74355 non-null  int64  
 3   PARTY        74355 non-null  object 
 4   CTYDIST      74355 non-null  object 
 5   SCHDIST      74355 non-null  int64  
 6   PRIMBALTYPE  0 non-null      float64
 7   FEDERAL      74355 non-null  object 
 8   AEVL         74355 non-null  object 
 9   YOB          74355 non-null  int64  
 10  voted_city   74355 non-null  int64  
dtypes: float64(1), int64(6), object(4)
memory usage: 6.2+ MB


In [34]:
voters2["FEDERAL"].value_counts()

FEDERAL
N    74095
Y      260
Name: count, dtype: int64

In [35]:
voters2.groupby("FEDERAL")["voted_city"].mean()

FEDERAL
N    0.311182
Y    0.003846
Name: voted_city, dtype: float64

In [36]:
#Removing federal only voters - law not behevior
voters3 = voters2[voters2["FEDERAL"] == "N"]

In [37]:
voters3.info()

<class 'pandas.core.frame.DataFrame'>
Index: 74095 entries, 0 to 74354
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   VOTERID      74095 non-null  int64  
 1   ZIPCODE      74095 non-null  int64  
 2   CPC          74095 non-null  int64  
 3   PARTY        74095 non-null  object 
 4   CTYDIST      74095 non-null  object 
 5   SCHDIST      74095 non-null  int64  
 6   PRIMBALTYPE  0 non-null      float64
 7   FEDERAL      74095 non-null  object 
 8   AEVL         74095 non-null  object 
 9   YOB          74095 non-null  int64  
 10  voted_city   74095 non-null  int64  
dtypes: float64(1), int64(6), object(4)
memory usage: 6.8+ MB


In [38]:
print(voters3["SCHDIST"])

0        28
1         3
2        28
3        28
4        28
         ..
74350     3
74351     3
74352     3
74353     3
74354    28
Name: SCHDIST, Length: 74095, dtype: int64


In [39]:
voters3["SCHDIST"].nunique()

4

In [40]:
print(voters3["AEVL"])

0        Y
1        Y
2        Y
3        N
4        Y
        ..
74350    Y
74351    N
74352    Y
74353    Y
74354    Y
Name: AEVL, Length: 74095, dtype: object


In [41]:
voters4 = voters3.copy()

In [42]:
voters4["AGE"] = 2024 - voters4["YOB"]

In [43]:
voters4 = voters4.drop(columns=["YOB"])

In [44]:
voters4["PRIMBALTYPE"].nunique()
voters4 = voters4.drop(columns = ["PRIMBALTYPE"])

In [45]:
voters4.info()

<class 'pandas.core.frame.DataFrame'>
Index: 74095 entries, 0 to 74354
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   VOTERID     74095 non-null  int64 
 1   ZIPCODE     74095 non-null  int64 
 2   CPC         74095 non-null  int64 
 3   PARTY       74095 non-null  object
 4   CTYDIST     74095 non-null  object
 5   SCHDIST     74095 non-null  int64 
 6   FEDERAL     74095 non-null  object
 7   AEVL        74095 non-null  object
 8   voted_city  74095 non-null  int64 
 9   AGE         74095 non-null  int64 
dtypes: int64(6), object(4)
memory usage: 6.2+ MB


In [46]:
voters4["CPC"].nunique()

209

In [47]:
print(voters4["CPC"])

0        105
1        872
2         98
3        162
4         12
        ... 
74350    408
74351    691
74352    523
74353    226
74354     12
Name: CPC, Length: 74095, dtype: int64


In [48]:
voters4["ZIPCODE"].nunique()

5

In [49]:
#precinct and school districts have been recently upadted in Tempe
voters5 = voters4.drop(columns = ["CPC", "SCHDIST"])

In [50]:
voters5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 74095 entries, 0 to 74354
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   VOTERID     74095 non-null  int64 
 1   ZIPCODE     74095 non-null  int64 
 2   PARTY       74095 non-null  object
 3   CTYDIST     74095 non-null  object
 4   FEDERAL     74095 non-null  object
 5   AEVL        74095 non-null  object
 6   voted_city  74095 non-null  int64 
 7   AGE         74095 non-null  int64 
dtypes: int64(4), object(4)
memory usage: 5.1+ MB


In [51]:
voters5["AEVL"] = voters5["AEVL"].map({"Y": 1, "N": 0})

In [52]:
voters5["AEVL"].value_counts()

AEVL
1    61028
0    13067
Name: count, dtype: int64

In [53]:
voters5 = voters5.drop(columns = ["CTYDIST"])

In [54]:
voters5["FEDERAL"].value_counts()

FEDERAL
N    74095
Name: count, dtype: int64

In [55]:
voters5 = voters5.drop(columns = ["FEDERAL"])

In [56]:
voters5.info()

<class 'pandas.core.frame.DataFrame'>
Index: 74095 entries, 0 to 74354
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   VOTERID     74095 non-null  int64 
 1   ZIPCODE     74095 non-null  int64 
 2   PARTY       74095 non-null  object
 3   AEVL        74095 non-null  int64 
 4   voted_city  74095 non-null  int64 
 5   AGE         74095 non-null  int64 
dtypes: int64(5), object(1)
memory usage: 4.0+ MB


#### Encoding

In [57]:
X = voters5.drop(columns=["voted_city", "VOTERID"])
y = voters5["voted_city"]

In [58]:
categorical_cols = ["PARTY", "ZIPCODE"]
numeric_cols = ["AGE", "AEVL"]

In [59]:
#Drop first to avoid collinearity
encoder = OneHotEncoder(drop="first", sparse_output=False)

In [60]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", encoder, categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)

In [61]:
X_encoded = preprocessor.fit_transform(X)

In [62]:
X.dtypes

ZIPCODE     int64
PARTY      object
AEVL        int64
AGE         int64
dtype: object

In [63]:
#stratify=y: keeps the same % of voters/nonvoters in train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.30, random_state=42, stratify=y
)

In [64]:
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression(max_iter=2000)
logit.fit(X_train, y_train)

LogisticRegression(max_iter=2000)

In [65]:
logit.n_iter_

array([326], dtype=int32)

In [66]:
y_pred = logit.predict(X_test)
y_prob = logit.predict_proba(X_test)[:, 1]

In [67]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7710648252283053
ROC AUC: 0.81145520954612
Confusion matrix:
 [[13490  1822]
 [ 3267  3650]]
              precision    recall  f1-score   support

           0       0.81      0.88      0.84     15312
           1       0.67      0.53      0.59      6917

    accuracy                           0.77     22229
   macro avg       0.74      0.70      0.72     22229
weighted avg       0.76      0.77      0.76     22229



In [68]:
coefs = logit.coef_[0]
odds_ratios = np.exp(coefs)

coef_table = pd.DataFrame({
    "feature_index": np.arange(len(coefs)),
    "coef": coefs,
    "odds_ratio": odds_ratios
}).sort_values("coef", ascending=False)

coef_table.head(15)   # strongest positive effects

,feature_index,coef,odds_ratio
13,13,1.432874,4.190728
10,10,0.460929,1.585547
8,8,0.192570,1.212362
9,9,0.104918,1.110620
12,12,0.051394,1.052737
4,4,-0.001473,0.998528
0,0,-0.061622,0.940238
1,1,-0.282163,0.754151
2,2,-0.329415,0.719345
11,11,-0.363426,0.695290


In [69]:
feature_names = preprocessor.get_feature_names_out()

In [70]:
coefs = logit.coef_[0]
odds_ratios = np.exp(coefs)

feature_names = preprocessor.get_feature_names_out()

coef_table = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs,
    "odds_ratio": odds_ratios
}).sort_values("coef", ascending=False)

coef_table.head(15)

,feature,coef,odds_ratio
13,num__AEVL,1.432874,4.190728
10,cat__ZIPCODE_85284,0.460929,1.585547
8,cat__ZIPCODE_85282,0.192570,1.212362
9,cat__ZIPCODE_85283,0.104918,1.110620
12,num__AGE,0.051394,1.052737
4,cat__PARTY_LIB,-0.001473,0.998528
0,cat__PARTY_DEM,-0.061622,0.940238
1,cat__PARTY_GRN,-0.282163,0.754151
2,cat__PARTY_IND,-0.329415,0.719345
11,cat__ZIPCODE_85288,-0.363426,0.695290


In [71]:
coef_table["feature"] = coef_table["feature"].str.replace("cat__", "")
coef_table["feature"] = coef_table["feature"].str.replace("num__", "")

In [72]:
coef_table.head(15)

,feature,coef,odds_ratio
13,AEVL,1.432874,4.190728
10,ZIPCODE_85284,0.460929,1.585547
8,ZIPCODE_85282,0.192570,1.212362
9,ZIPCODE_85283,0.104918,1.110620
12,AGE,0.051394,1.052737
4,PARTY_LIB,-0.001473,0.998528
0,PARTY_DEM,-0.061622,0.940238
1,PARTY_GRN,-0.282163,0.754151
2,PARTY_IND,-0.329415,0.719345
11,ZIPCODE_85288,-0.363426,0.695290


In [73]:
voters6 = voters5.drop(columns=["VOTERID"])

In [74]:
voters6["PARTY_clean"] = voters6["PARTY"]

voters6["AGE_c"] = voters6["AGE"] - voters6["AGE"].mean()

voters6.loc[~voters6["PARTY"].isin(["DEM", "REP", "IND"]), "PARTY_clean"] = "OTHER"

In [75]:
import statsmodels.formula.api as smf

model = smf.logit(
    formula="voted_city ~ AGE_c + AEVL + C(ZIPCODE, Treatment(reference=85281)) + C(PARTY_clean, Treatment(reference='IND'))",
    data=voters6
).fit(maxiter=5000)

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.484356
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:             voted_city   No. Observations:                74095
Model:                          Logit   Df Residuals:                    74085
Method:                           MLE   Df Model:                            9
Date:                Fri, 20 Mar 2026   Pseudo R-squ.:                  0.2188
Time:                        19:44:23   Log-Likelihood:                -35888.
converged:                       True   LL-Null:                       -45942.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                          coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------
Intercept                 

In [76]:
#85284 is sig. larger than 85282(the closest to the refrence which is the largest zipcode)
model.t_test("C(ZIPCODE, Treatment(reference=85281))[T.85284] - C(ZIPCODE, Treatment(reference=85281))[T.85282] = 0")

<class 'statsmodels.stats.contrast.ContrastResults'>
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.2619      0.026      9.976      0.000       0.210       0.313

In [77]:
voters6["AGE_std"] = (voters6["AGE"] - voters6["AGE"].mean()) / voters6["AGE"].std()

In [78]:
model = smf.logit(
    formula="voted_city ~ AGE_std + AEVL + C(ZIPCODE, Treatment(reference=85281)) + C(PARTY_clean, Treatment(reference='IND'))",
    data=voters6
).fit(maxiter=5000)

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.484356
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:             voted_city   No. Observations:                74095
Model:                          Logit   Df Residuals:                    74085
Method:                           MLE   Df Model:                            9
Date:                Fri, 20 Mar 2026   Pseudo R-squ.:                  0.2188
Time:                        19:44:23   Log-Likelihood:                -35888.
converged:                       True   LL-Null:                       -45942.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                          coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------
Intercept                 

In [79]:
zip_code = pd.read_csv('ZipCodeData.csv')

In [80]:
print(zip_code["Label (Grouping)"])

0                                    SEX AND AGE
1                               Total population
2                                           Male
3                                         Female
4              Sex ratio (males per 100 females)
                         ...                    
108                          Total housing units
109               CITIZEN, VOTING AGE POPULATION
110              Citizen, 18 and over population
111                                         Male
112                                       Female
Name: Label (Grouping), Length: 113, dtype: object


In [81]:
zip_code

,Label (Grouping),ZCTA5 85008!!Estimate,ZCTA5 85008!!Margin of Error,ZCTA5 85008!!Percent,ZCTA5 85008!!Percent Margin of Error,ZCTA5 85044!!Estimate,ZCTA5 85044!!Margin of Error,ZCTA5 85044!!Percent,ZCTA5 85044!!Percent Margin of Error,ZCTA5 85202!!Estimate,...,ZCTA5 85282!!Percent,ZCTA5 85282!!Percent Margin of Error,ZCTA5 85283!!Estimate,ZCTA5 85283!!Margin of Error,ZCTA5 85283!!Percent,ZCTA5 85283!!Percent Margin of Error,ZCTA5 85284!!Estimate,ZCTA5 85284!!Margin of Error,ZCTA5 85284!!Percent,ZCTA5 85284!!Percent Margin of Error
0,SEX AND AGE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Total population,"59,716","±2,956","59,716",(X),"39,347","±2,050","39,347",(X),"38,750",...,"53,624",(X),"47,459","±2,022","47,459",(X),"19,492","±1,459","19,492",(X)
2,Male,"31,439","±1,777",52.6%,±1.5,"19,143","±1,262",48.7%,±1.9,"20,115",...,51.6%,±1.8,"23,788","±1,303",50.1%,±2.1,"10,392","±1,068",53.3%,±2.2
3,Female,"28,277","±1,689",47.4%,±1.5,"20,204","±1,258",51.3%,±1.9,"18,635",...,48.4%,±1.8,"23,671","±1,502",49.9%,±2.1,"9,100",±573,46.7%,±2.2
4,Sex ratio (males per 100 females),111.2,±6.7,(X),(X),94.7,±7.1,(X),(X),107.9,...,(X),(X),100.5,±8.3,(X),(X),114.2,±9.9,(X),(X)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,Total housing units,"24,630",±814,(X),(X),"18,731",±596,(X),(X),"18,154",...,(X),(X),"20,719",±801,(X),(X),"7,644",±409,(X),(X)
109,"CITIZEN, VOTING AGE POPULATION",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
110,"Citizen, 18 and over population","35,220","±1,697","35,220",(X),"30,776","±1,506","30,776",(X),"28,407",...,"42,438",(X),"35,171","±1,508","35,171",(X),"14,998","±1,108","14,998",(X)
111,Male,"18,990","±1,017",53.9%,±1.8,"14,670",±960,47.7%,±2.0,"14,420",...,51.5%,±2.0,"17,694","±1,090",50.3%,±1.9,"7,858",±799,52.4%,±2.2


In [82]:
# 1) Keep only the ZIPs you care about
important_zips = ["85281", "85282", "85283", "85284", "85288"]

cols_to_keep = ["Label (Grouping)"]
for z in important_zips:
    cols_to_keep += [c for c in zip_code.columns if z in c]

zip_filtered = zip_code[cols_to_keep]


# 2) Clean column names so they're easier to work with
#    Example: "ZCTA5 85284!!Estimate" -> "85284_Estimate"
#             "ZCTA5 85284!!Margin of Error" -> "85284_MOE"
zip_filtered.columns = [
    c.replace("ZCTA5 ", "")
     .replace("!!Estimate", "_Estimate")
     .replace("!!Margin of Error", "_MOE")
    for c in zip_filtered.columns
]


# 3) Make the row labels (e.g., "Male", "Female") the index
zip_filtered = zip_filtered.set_index("Label (Grouping)")


# 4) Transpose so ZIPs become rows and categories become columns
zip_t = zip_filtered.T


# 5) Split the row index into ZIP and Type (Estimate vs MOE)
zip_t["ZIP"] = zip_t.index.str.split("_").str[0]
zip_t["Type"] = zip_t.index.str.split("_").str[1]


# 6) Pivot so "Estimate" and "MOE" become columns (instead of separate rows)
zip_final = zip_t.pivot_table(index="ZIP", columns="Type", aggfunc="first")


# 7) Flatten the multi-index columns into single strings like "Male_Estimate"
zip_final.columns = ["_".join(col).strip() for col in zip_final.columns]


# Optional: drop rows that are completely empty
zip_final = zip_final.dropna(how="all")

zip_final.head()

,Total housing units_Estimate,Total housing units_MOE,"Citizen, 18 and over population_Estimate","Citizen, 18 and over population_MOE",Total population_Estimate,Total population_MOE,Total population_Estimate,Total population_MOE,Total population_Estimate,Total population_MOE,...,Samoan_Estimate,Samoan_MOE,Somali_Estimate,Somali_MOE,"Two races excluding Some Other Race, and three or more races_Estimate","Two races excluding Some Other Race, and three or more races_MOE",Two races including Some Other Race_Estimate,Two races including Some Other Race_MOE,Vietnamese_Estimate,Vietnamese_MOE
ZIP,,,,,,,,,,,,,,,,,,,,,
85281,"33,978","±1,089","56,771","±1,988","72,762","±2,163","72,762","±2,163","72,762","±2,163",...,37,±49,0,±33,"4,894",±973,637,±391,471,±277
85282,"25,593",±970,"42,438","±1,750","53,624","±2,259","53,624","±2,259","53,624","±2,259",...,11,±19,560,±610,"2,820",±610,711,±362,525,±235
85283,"20,719",±801,"35,171","±1,508","47,459","±2,022","47,459","±2,022","47,459","±2,022",...,0,±30,278,±288,"1,763",±509,443,±179,543,±317
85284,"7,644",±409,"14,998","±1,108","19,492","±1,459","19,492","±1,459","19,492","±1,459",...,0,±22,0,±22,468,±183,115,±76,103,±89


In [83]:
zip_turnout = voters5.groupby("ZIPCODE")["voted_city"].mean()

In [84]:
zip_turnout

ZIPCODE
85281    0.218251
85282    0.332373
85283    0.321150
85284    0.429950
85288    0.171284
Name: voted_city, dtype: float64

In [85]:
zip_filtered = zip_final.copy()
zip_filtered = zip_filtered.loc[:, ~zip_filtered.columns.str.contains("Margin")]
zip_filtered = zip_filtered.loc[:, ~zip_filtered.columns.str.contains("MOE")]

In [86]:

zip_filtered = zip_filtered.loc[:, ~zip_filtered.columns.duplicated()]

In [87]:
zip_filtered

,Total housing units_Estimate,"Citizen, 18 and over population_Estimate",Total population_Estimate,10 to 14 years_Estimate,15 to 19 years_Estimate,16 years and over_Estimate,18 years and over_Estimate,20 to 24 years_Estimate,21 years and over_Estimate,25 to 34 years_Estimate,...,Other American Indian and Alaska Native_Estimate,Other Asian_Estimate,Other Black or African American_Estimate,Other Native Hawaiian and Other Pacific Islander_Estimate,Other White_Estimate,Samoan_Estimate,Somali_Estimate,"Two races excluding Some Other Race, and three or more races_Estimate",Two races including Some Other Race_Estimate,Vietnamese_Estimate
ZIP,,,,,,,,,,,,,,,,,,,,,
85281,"33,978","56,771","72,762","1,932","10,234","66,472","65,931","19,446","52,262","18,619",...,608,"1,322","1,301",178,"28,458",37,0,"4,894",637,471
85282,"25,593","42,438","53,624","2,163","2,133","45,843","45,143","6,513","43,123","13,251",...,738,364,"1,321",0,"21,042",11,560,"2,820",711,525
85283,"20,719","35,171","47,459","2,317","2,797","39,651","38,485","4,519","36,609","8,877",...,"2,177",341,"1,328",168,"17,927",0,278,"1,763",443,543
85284,"7,644","14,998","19,492","1,197","1,412","16,455","15,873","1,287","15,177","1,917",...,90,203,403,0,"8,834",0,0,468,115,103


In [91]:
zip_filtered.columns

Index(['Total housing units_Estimate',
       'Citizen, 18 and over population_Estimate', 'Total population_Estimate',
       '10 to 14 years_Estimate', '15 to 19 years_Estimate',
       '16 years and over_Estimate', '18 years and over_Estimate',
       '20 to 24 years_Estimate', '21 years and over_Estimate',
       '25 to 34 years_Estimate', '35 to 44 years_Estimate',
       '45 to 54 years_Estimate', '5 to 9 years_Estimate',
       '55 to 59 years_Estimate', '60 to 64 years_Estimate',
       '62 years and over_Estimate', '65 to 74 years_Estimate',
       '65 years and over_Estimate', '75 to 84 years_Estimate',
       '85 years and over_Estimate',
       'American Indian and Alaska Native_Estimate', 'Asian_Estimate',
       'Black or African American_Estimate', 'Female_Estimate',
       'Hispanic or Latino (of any race)_Estimate', 'Male_Estimate',
       'Median age (years)_Estimate',
       'Native Hawaiian and Other Pacific Islander_Estimate',
       'Not Hispanic or Latino_Estimate